# Python Warm-Up: Foundations for Acrobot (pygame + Gymnasium)

---
## Section 1: Loops, State, and "Feeding Output Back In"

### `while True` + `break` vs. `for`

Use `for` when you know the number of repetitions ahead of time (e.g. "try 20 evaluation episodes").
Use `while True:` + `break` when you don't know how long something will run and instead want to stop the moment a *condition* becomes true (e.g. "run until the episode ends").

```python
while True:
    # do something
    if some_condition:
        break
```

### State feeding into itself

```python
state = start
while True:
    state = update(state)   # this iteration's output becomes next iteration's input
    if done(state):
        break
```

This is exactly the shape of `state = next_state` you'll see later in the Q-learning loop, and of updating a ball's position/velocity each frame in pygame.


### Live demo

Run this cell together. A "ball" starts at position 0 with velocity 2. Each tick, position updates using its own previous value. We stop the moment it crosses 100.

In [ ]:
position = 0
velocity = 2
steps = 0

while True:
    position += velocity   # <-- output feeds back in as next input
    steps += 1

    if position > 100:
        break

print(f"Crossed 100 after {steps} steps. Final position: {position}")


### 🧪 Exercise 1

Write a loop that starts a ball at `position = 0` with `velocity = 3`.

- Each tick, update `position`.
- Every 5th step, increase `velocity` by 1 (the ball speeds up — like gravity, sort of).
- Stop the loop once `position > 50`, and print how many steps it took.

Use a `while True:` + `break` — no `for` loop here, since we don't know in advance how many steps it'll take.


In [3]:
# TODO: your code here

position = 0
velocity = 3
steps = 0

while True:
    position = position + velocity # position += velocity
    steps += 1
    if steps % 5 == 0: # check if steps is a multiple of 5 (when you divide steps by 5, the remainder is zero)
        velocity += 1
    if position > 50:
        break

print(steps)


14


In [ ]:
for i in range(30):
    print(i % 5)
    # if i % 3 == 0:
    #     print("fizz")
    # elif i % 5 == 0:
    #     print("buzz")
    # else:
    #     print(i)

<details>
<summary>▶ Solution (click to reveal)</summary>

```python
position = 0
velocity = 3
steps = 0

while True:
    position = position + velocity
    steps += 1

    if steps % 5 == 0:
        velocity += 1

    if position > 50:
        break

print(f"Done after {steps} steps. Final position: {position}, final velocity: {velocity}")
```
</details>


---
## Section 2: Functions, Defaults, and Multi-Value Returns

Two patterns you'll need:

**1. Default arguments** — lets you call a function with or without overriding certain values:
```python
def step(state, velocity=1):
    ...
```

**2. Returning multiple values, then unpacking them** - a function can return several things at once as a tuple, and you can unpack them directly into named variables:
```python
def move(x, y):
    return x + 1, y + 2

new_x, new_y = move(3, 4)   # tuple unpacking — same trick as `a, b = 1, 2`
```

You'll see this shape in Gymnasium: `observation, reward, terminated, truncated, info = env.step(action)`; one function call, five values back, all unpacked in one line.

### Combining stop conditions with `or`

```python
done = terminated or truncated   # True if EITHER is True
```


### Live demo

`move()` shows both patterns from above at once: it has a **default argument** (`step=1`) and it **returns two values** that we unpack in the loop.

Watch how the loop looks almost identical to Section 1's ball loop. The only difference is that the update logic now lives inside a function instead of being written inline.

In [ ]:
def move(x, step=1):
    new_x = x + step
    crossed_zero = new_x > 0 and x <= 0  # True the instant we cross from <=0 to >0
    return new_x, crossed_zero


x = -3
for _ in range(5):
    x, crossed = move(x)  # tuple unpacking, same as env.step() later
    print(f"x={x}, crossed_zero={crossed}")

# calling with the default vs. overriding it:
x2, _ = move(10)  # uses default step=1
x3, _ = move(10, step=5)  # overrides the default
print("x2 (default step):", x2)
print("x3 (step=5):", x3)

### 🧪 Exercise 2

Now extend the demo above: instead of `move(x, step=1)`, refactor the **ball simulation** from Section 1 into a function the same way.

Write `def step(position, velocity):` that:
- returns the **new position**, and
- a boolean `done` that's `True` once `position > 50`

(just like `move()` returned `new_x` and `crossed_zero` together)

Then write a loop that calls `step(...)`, unpacks the result — same pattern as `x, crossed = move(x)` — updates `position`, and breaks when `done` is `True`.

In [5]:

position = 0
velocity = 3
steps = 0

while True:
    position = position + velocity # position += velocity
    steps += 1
    if steps % 5 == 0: # check if steps is a multiple of 5 (when you divide steps by 5, the remainder is zero)
        velocity += 1
    if position > 50:
        break

print(steps)

#################################################################
def step(position, velocity):
    new_position = position + velocity
    # if position > 50:
    #     done = True
    # else:
    #     done = False

    done = position > 50
    return new_position, done

# TODO: write the loop that calls step() and unpacks its return value
position = 0
velocity = 3
steps = 0
while True:
    position, done = step(position, velocity)
    steps += 1
    if steps % 5 == 0: # check if steps is a multiple of 5 (when you divide steps by 5, the remainder is zero)
        velocity += 1
    if done:
        break
print(steps)

14
15


<details>
<summary>▶ Solution (click to reveal)</summary>

```python
def step(position, velocity):
    new_position = position + velocity
    done = new_position > 50
    return new_position, done

position = 0
velocity = 3
steps = 0

while True:
    position, done = step(position, velocity)
    steps += 1
    if done:
        break

print(f"Done after {steps} steps. Final position: {position}")
```
</details>


---
## Section 3: Dictionaries as Lookup Tables, Tuples as Keys

### The "look up, or create if missing" pattern

```python
Q = {}                     # empty dictionary: nothing known yet

if state not in Q:
    Q[state] = 0            # create an entry the first time we see this state

Q[state] += 1               # now safe to update it
```

This lets you store information **only for states you've seen**, instead of pre-allocating a giant table for every possible state up front (which is often impossible; imagine trying to list every possible sensor reading!).

### Why tuples (not lists) as dictionary keys

Dictionary keys must be **hashable**, essentially, "unchangeable" (immutable). Tuples are immutable (`(2, 1, 4)`), so they work as keys. Lists are mutable (`[2, 1, 4]` can change after creation), so Python won't allow them as keys at all; you'll get an error if you try.

```python
position = (2, 1, 4)        # tuple: OK as a dict key
# position = [2, 1, 4]      # list: NOT allowed as a dict key
```


### Live demo

We'll build a dictionary that counts how many times we visit each `(x, y)` grid coordinate, creating the entry the first time we see it.

In [7]:
import random

visit_counts = {}   # maps (x, y) -> number of visits

for _ in range(20):
    coord = (random.randint(0, 2), random.randint(0, 2))   # a random (x, y) tuple

    if coord not in visit_counts:
        visit_counts[coord] = 0

    visit_counts[coord] += 1

for coord, count in visit_counts.items():
    print(f"{coord}: visited {count} time(s)")
print(visit_counts)


(1, 1): visited 5 time(s)
(0, 1): visited 4 time(s)
(0, 0): visited 3 time(s)
(2, 0): visited 3 time(s)
(1, 2): visited 1 time(s)
(2, 1): visited 2 time(s)
(1, 0): visited 1 time(s)
(0, 2): visited 1 time(s)
{(1, 1): 5, (0, 1): 4, (0, 0): 3, (2, 0): 3, (1, 2): 1, (2, 1): 2, (1, 0): 1, (0, 2): 1}


### 🧪 Exercise 3

Write a function `get_or_create(table, key)` that:
- returns `table[key]` if `key` is already in `table`
- otherwise, creates `table[key] = []` (an empty list), stores it, and returns it

Then use it to build a dictionary that groups a list of `(x, y)` points by which "bucket" they fall in, where the bucket is `(x // 10, y // 10)` (integer division, like a coarse grid). Append each point to the list for its bucket.


In [ ]:
points = [(3, 4), (12, 7), (5, 5), (13, 19), (25, 3), (4, 8), (11, 11)]

def get_or_create(table, key):
    if key not in table:
        table[key] = []
    return table[key]


buckets = {}

# TODO: loop over `points`, compute each point's bucket key,
# use get_or_create() to fetch (or create) that bucket's list,
# and append the point to it.
for x, y in points:
    bucket_key = (x//10, y//10)

print(buckets)


<details>
<summary>▶ Solution (click to reveal)</summary>

```python
points = [(3, 4), (12, 7), (5, 5), (13, 19), (25, 3), (4, 8), (11, 11)]

def get_or_create(table, key):
    if key not in table:
        table[key] = []
    return table[key]

buckets = {}

for x, y in points:
    bucket_key = (x // 10, y // 10)
    bucket_list = get_or_create(buckets, bucket_key)
    bucket_list.append((x, y))

print(buckets)
```
</details>


---
## Section 4: NumPy Essentials

Only the handful of operations you'll actually see later:

| Function | What it does | Where you'll see it |
|---|---|---|
| `np.zeros(n)` | Array of `n` zeros | Initializing Q-values for a new state |
| `np.argmax(array)` | Index of the largest value | "Which action has the best Q-value?" |
| `np.linspace(low, high, n)` | `n` evenly spaced numbers between `low` and `high` | Defining bin edges for discretization |
| `np.digitize(value, edges)` | Which bin a value falls into, given edges | Turning a continuous reading into a bin index |


In [ ]:
import numpy as np

# np.zeros -- a fresh row of "unknown" values, one per action
q_values = np.zeros(3)
print("q_values:", q_values)

# np.argmax -- which index holds the largest value?
q_values = np.array([1.2, 5.7, 3.3])
best_action = np.argmax(q_values)
print("best_action:", best_action)


### Live demo: discretization

This is the trick that turns a continuous number into a finite "bucket", exactly what we'll need for a Q-table later, since Q-tables need a finite set of rows.

In [ ]:
# Define 4 bins between -1 and 1 using 3 interior edges
edges = np.linspace(-1, 1, 3)
print("bin edges:", edges)

# Which bin does each value fall into?
for value in [-0.9, -0.1, 0.4, 0.95]:
    bin_index = np.digitize(value, edges)
    print(f"value {value:>5} -> bin {bin_index}")


### 🧪 Exercise 4

You have a fake sensor that reports a continuous value between -5 and 5.

1. Use `np.linspace` to create 4 bin edges between -5 and 5.
2. Write a function `discretize(value)` that uses `np.digitize` to return which bin the value falls into.
3. Combine it with what you learned in Section 3: build a dictionary keyed by bin index that counts how many times each bin has been hit, for the sample readings given below.


In [ ]:
readings = [-4.8, -1.2, 0.5, 3.9, 4.9, -0.1, 2.2, -3.3, 1.1, 0.0]

# TODO: create bin edges with np.linspace

# TODO: define discretize(value) using np.digitize

# TODO: build a dict counting how many readings fall in each bin
bin_counts = {}



<details>
<summary>▶ Solution (click to reveal)</summary>

```python
readings = [-4.8, -1.2, 0.5, 3.9, 4.9, -0.1, 2.2, -3.3, 1.1, 0.0]

edges = np.linspace(-5, 5, 4)

def discretize(value):
    return int(np.digitize(value, edges))

bin_counts = {}
for reading in readings:
    b = discretize(reading)
    if b not in bin_counts:
        bin_counts[b] = 0
    bin_counts[b] += 1

print(bin_counts)
```
</details>


---
## Section 5: Trig Primer + Preview of What's Next

Gymnasium hides the physics of Acrobot from you entirely — you just get numbers back. But in **pygame**, you're building the simulation yourself, so you need a little trigonometry to place the arm segments on screen.

### Placing the end of a rotating segment

If a segment of length `L` is anchored at `(origin_x, origin_y)` and rotated to angle `theta` (in radians), the free end of the segment is at:

```python
import math

end_x = origin_x + L * math.cos(theta)
end_y = origin_y + L * math.sin(theta)
```

For Acrobot's *second* link, you just repeat this, anchoring at the *first* link's end point instead of a fixed origin.


In [ ]:
import math

origin_x, origin_y = 300, 200
L1 = 100
theta1 = math.radians(40)   # 40 degrees, converted to radians

end_x = origin_x + L1 * math.cos(theta1)
end_y = origin_y + L1 * math.sin(theta1)

print(f"Link 1 end point: ({end_x:.1f}, {end_y:.1f})")


### The pygame game loop (preview only)

Notice the shape: this is the *exact same loop pattern* from Section 1, just with pygame-specific pieces slotted in.

```python
import pygame

pygame.init()
screen = pygame.display.set_mode((600, 400))
clock = pygame.time.Clock()
running = True

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # update state here (angles, velocities...)

    screen.fill((255, 255, 255))
    # draw here

    pygame.display.flip()
    clock.tick(60)   # cap at 60 frames per second

pygame.quit()
```

**The takeaway:** whether it's `while True: ... break` in a plain Python loop, `while running:` in pygame, or the training loop in the Acrobot notebook — it's always *poll or step → update state → check a stopping condition → repeat*. Everything we did today was in service of making that loop, and the tools inside it (functions, dicts, NumPy), feel natural.
